In [2]:
# ENVIRONMENT SETUP

!pip install biopython
!apt-get update -qq
!apt-get install -y minimap2

import os
import math
import random
import gzip
import subprocess
import time
from Bio import SeqIO
from google.colab import drive
from collections import Counter, defaultdict

drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.1 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  minimap2
0 upgraded, 1 newly installed, 0 to remove and 28 not upgraded.
Need to get 381 kB of archives.
After this operation, 497 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 minimap2 amd64 2.24+dfsg-2 [381 kB]
Fetched 381 kB in 0s (4,581 kB/s)
Selecting previously unselected package minimap2.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../minimap2_2.24+dfsg-2_amd64.deb ...
Unpacking minimap2 (2.24+dfsg-2) ...
Setting up minimap2 (2.24+dfsg-2) ...
Processing triggers for man-d

In [3]:
# METAGENOME CREATION

PATH_READS = "/content/drive/Shareddrives/BIOINFORMATICS/reads"

READS_PER_BACTERIA = 5000
OUTPUT_NAME = "metagenome_final.fasta"


def generate_metagenome():
    all_reads = []

    try:
        source_files = [f for f in os.listdir(PATH_READS) if f.endswith(".fastq.gz")]
    except FileNotFoundError:
        print(f"ERROR: Cannot find the path: {PATH_READS}")
        return

    if not source_files:
        print("No .fastq.gz files found in the directory.")
        return

    for file_name in source_files:
        tag = file_name.split(".")[0]
        full_path = os.path.join(PATH_READS, file_name)

        print(f"Processing {tag}...")

        with gzip.open(full_path, "rt") as handle:
            count = 0

            for record in SeqIO.parse(handle, "fastq"):
                if count >= READS_PER_BACTERIA:
                    break

                record.id = f"{tag}|{record.id}"
                record.description = ""

                all_reads.append(record)
                count += 1

            print(f"{count} reads extracted.")

    print(f"Shuffling {len(all_reads)} reads randomly...")
    random.shuffle(all_reads)

    SeqIO.write(all_reads, OUTPUT_NAME, "fasta")

    print(f"'{OUTPUT_NAME}' has been generated.")


generate_metagenome()


# DATASET SUMMARY

species_counts = Counter()
species_bases = defaultdict(int)
species_fasta_bytes = defaultdict(int)

total_count = 0

for record in SeqIO.parse(OUTPUT_NAME, "fasta"):
    species = record.id.split("|")[0]
    seq = str(record.seq)

    species_counts[species] += 1
    species_bases[species] += len(seq)

    fasta_entry = f">{record.id}\n{seq}\n"
    species_fasta_bytes[species] += len(fasta_entry.encode("utf-8"))

    total_count += 1

total_file_size_mb = os.path.getsize(OUTPUT_NAME) / (1024 * 1024)

print("\nMETAGENOME DATASET SUMMARY")
print(f"Total reads: {total_count}")
print(f"Total FASTA file size: {total_file_size_mb:.2f} MB")

print("\nPER-SPECIES SUMMARY")

for species in sorted(species_counts):
    size_mb = species_fasta_bytes[species] / (1024 * 1024)
    mean_length = species_bases[species] / species_counts[species]

    print(f"\n{species}")
    print(f"Reads: {species_counts[species]}")
    print(f"Mean read length: {mean_length:.2f} bp")
    print(f"Approx. FASTA size: {size_mb:.2f} MB")

Processing salmonella_enterica...
5000 reads extracted.
Processing escherichia_coli...
5000 reads extracted.
Processing streptococcus_infantis...
5000 reads extracted.
Processing vibrio_mimicus...
5000 reads extracted.
Processing bacillus_lentus...
5000 reads extracted.
Shuffling 25000 reads randomly...
'metagenome_final.fasta' has been generated.

METAGENOME DATASET SUMMARY
Total reads: 25000
Total FASTA file size: 99.60 MB

PER-SPECIES SUMMARY

bacillus_lentus
Reads: 5000
Mean read length: 4457.91 bp
Approx. FASTA size: 21.71 MB

escherichia_coli
Reads: 5000
Mean read length: 3637.63 bp
Approx. FASTA size: 17.80 MB

salmonella_enterica
Reads: 5000
Mean read length: 3806.56 bp
Approx. FASTA size: 18.62 MB

streptococcus_infantis
Reads: 5000
Mean read length: 5273.05 bp
Approx. FASTA size: 25.63 MB

vibrio_mimicus
Reads: 5000
Mean read length: 2895.11 bp
Approx. FASTA size: 14.25 MB


In [4]:
# K-MER CLASSIFICATION

PATH_REFS = "/content/drive/Shareddrives/BIOINFORMATICS/references/"
METAGENOME_FILE = "metagenome_final.fasta"

K = 5
VALID_BASES = {"A", "C", "G", "T"}


def get_kmer_distribution(sequence, k):
    sequence = str(sequence).upper()

    if len(sequence) < k:
        return {}

    counts = Counter()

    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]

        if set(kmer).issubset(VALID_BASES):
            counts[kmer] += 1 #sliding window over the sequence

    total = sum(counts.values())

    if total == 0:
        return {}

    return {kmer: count / total for kmer, count in counts.items()}


def vector_norm(distribution):
    return math.sqrt(sum(value * value for value in distribution.values()))


def cosine_similarity(read_dist, read_norm, ref_dist, ref_norm):
    if not read_dist or read_norm == 0.0 or ref_norm == 0.0:
        return 0.0

    dot_product = sum(value * ref_dist.get(kmer, 0.0)
                      for kmer, value in read_dist.items())

    return dot_product / (read_norm * ref_norm)

kmer_start_time = time.time()

print(f"Indexing reference genomes with K={K}...")

ref_profiles = {}
ref_norms = {}

ref_files = [f for f in os.listdir(PATH_REFS)
             if f.endswith(".fasta") or f.endswith(".fna")]

for file in ref_files:
    ref_name = file.split(".")[0]
    ref_path = os.path.join(PATH_REFS, file)

    full_counts = Counter()

    for record in SeqIO.parse(ref_path, "fasta"):
        seq = str(record.seq).upper()

        if len(seq) < K:
            continue

        for i in range(len(seq) - K + 1):
            kmer = seq[i:i+K]

            if set(kmer).issubset(VALID_BASES):
                full_counts[kmer] += 1

    total = sum(full_counts.values())

    if total == 0:
        ref_profiles[ref_name] = {}
        ref_norms[ref_name] = 0.0
        continue

    ref_dist = {kmer: count / total for kmer, count in full_counts.items()}

    ref_profiles[ref_name] = ref_dist
    ref_norms[ref_name] = vector_norm(ref_dist)

    print(f"Indexed {ref_name} ({len(ref_dist)} unique k-mers)")


print(f"\nClassifying reads from {METAGENOME_FILE}...")

kmer_results = []
correct_assignments = 0
total_reads = 0

kmer_predicted_counts = defaultdict(int)

for read in SeqIO.parse(METAGENOME_FILE, "fasta"):
    total_reads += 1

    read_dist_fwd = get_kmer_distribution(read.seq, K)
    read_dist_rev = get_kmer_distribution(read.seq.reverse_complement(), K)

    read_norm_fwd = vector_norm(read_dist_fwd)
    read_norm_rev = vector_norm(read_dist_rev)

    best_match = None
    best_score = -1.0

    for ref_name, ref_dist in ref_profiles.items():
        score_fwd = cosine_similarity(read_dist_fwd, read_norm_fwd,
                                      ref_dist, ref_norms[ref_name])

        score_rev = cosine_similarity(read_dist_rev, read_norm_rev,
                                      ref_dist, ref_norms[ref_name])

        score = max(score_fwd, score_rev)

        if score > best_score:
            best_score = score
            best_match = ref_name

    true_label = read.id.split("|")[0]

    if best_match == true_label:
        correct_assignments += 1

    kmer_predicted_counts[best_match] += 1

    kmer_results.append({
        "read_id": read.id,
        "predicted": best_match,
        "actual": true_label,
        "score": best_score
    })


accuracy = (correct_assignments / total_reads) * 100 if total_reads > 0 else 0.0

print("\nK-MER CLASSIFICATION REPORT")
print(f"Total reads: {total_reads}")
print(f"Correctly classified: {correct_assignments}")
print(f"Accuracy: {accuracy:.2f}%")

print("\nK-MER METAGENOMIC SAMPLE COMPOSITION")
for genome, count in sorted(kmer_predicted_counts.items()):
    print(f"{genome}: {count} reads")

kmer_end_time = time.time()

kmer_runtime = kmer_end_time - kmer_start_time

print(f"\nK-MER Runtime: {kmer_runtime:.2f} seconds")

Indexing reference genomes with K=5...
Indexed escherichia_coli (1024 unique k-mers)
Indexed streptococcus_infantis (1024 unique k-mers)
Indexed salmonella_enterica (1024 unique k-mers)
Indexed vibrio_mimicus (1024 unique k-mers)
Indexed bacillus_lentus (1024 unique k-mers)

Classifying reads from metagenome_final.fasta...

K-MER CLASSIFICATION REPORT
Total reads: 25000
Correctly classified: 19796
Accuracy: 79.18%

K-MER METAGENOMIC SAMPLE COMPOSITION
bacillus_lentus: 6835 reads
escherichia_coli: 3872 reads
salmonella_enterica: 4621 reads
streptococcus_infantis: 4476 reads
vibrio_mimicus: 5196 reads

K-MER Runtime: 269.75 seconds


In [5]:
# SIMPLE MINIMAP2 TEST: one reference + one reads file

PATH_READS = "/content/drive/Shareddrives/BIOINFORMATICS/reads"
PATH_REFS = "/content/drive/Shareddrives/BIOINFORMATICS/references"

test_ref = "/content/drive/Shareddrives/BIOINFORMATICS/references/escherichia_coli.fasta"
test_reads = "/content/drive/Shareddrives/BIOINFORMATICS/reads/escherichia_coli.fastq.gz"
test_output = "minimap2_test_output.paf"

command = f"minimap2 -x map-pb {test_ref} {test_reads} > {test_output}"

print("Reference:", test_ref)
print("Reads:", test_reads)
print("Command:", command)

subprocess.run(command, shell=True)

print("\nFirst lines of minimap2 output:")
with open(test_output, "r") as f:
    for i, line in enumerate(f):
        if i == 5:
            break
        print(line.strip())

Reference: /content/drive/Shareddrives/BIOINFORMATICS/references/escherichia_coli.fasta
Reads: /content/drive/Shareddrives/BIOINFORMATICS/reads/escherichia_coli.fastq.gz
Command: minimap2 -x map-pb /content/drive/Shareddrives/BIOINFORMATICS/references/escherichia_coli.fasta /content/drive/Shareddrives/BIOINFORMATICS/reads/escherichia_coli.fastq.gz > minimap2_test_output.paf

First lines of minimap2 output:
m151222_181723_00127_c100958512550000001823217206251630_s1_p0/9/9137_16678	7541	59	2684	-	NC_000913.3	4641652	1246700	1248463	558	2627	60	tp:A:P	cm:i:44	s1:i:406	s2:i:101	dv:f:0.0091	rl:i:0
m151222_181723_00127_c100958512550000001823217206251630_s1_p0/9/9137_16678	7541	6390	6518	+	NC_000913.3	4641652	1246458	1246557	55	128	11	tp:A:P	cm:i:3	s1:i:53	s2:i:0	dv:f:0.0192	rl:i:0
m151222_181723_00127_c100958512550000001823217206251630_s1_p0/10/15809_17216	1407	32	1380	+	NC_000913.3	4641652	2847780	2849091	775	1353	60	tp:A:P	cm:i:56	s1:i:748	s2:i:0	dv:f:0.0027	rl:i:0
m151222_181723_00127_c10

In [6]:
# MINIMAP2 ALIGNMENT-BASED CLASSIFICATION

PATH_REFS = "/content/drive/Shareddrives/BIOINFORMATICS/references/"
METAGENOME_FILE = "metagenome_final.fasta"

MINIMAP_OUTPUT_DIR = "minimap2_outputs"
os.makedirs(MINIMAP_OUTPUT_DIR, exist_ok=True)

ref_files = [f for f in os.listdir(PATH_REFS)
             if f.endswith(".fasta") or f.endswith(".fna")]

minimap_best_results = {}#stores the best alignment result per read across all references
minimap_start_time = time.time()

print("Running minimap2 for each reference genome...")

for ref_file in ref_files:
    ref_name = ref_file.split(".")[0]
    ref_path = os.path.join(PATH_REFS, ref_file)

    output_paf = os.path.join(MINIMAP_OUTPUT_DIR, f"{ref_name}.paf")

    command = f"minimap2 -x map-pb {ref_path} {METAGENOME_FILE} > {output_paf}"

    print(f"Running minimap2 for {ref_name}...")
    subprocess.run(command, shell=True)

    with open(output_paf, "r") as paf_file:
        for line in paf_file:
            fields = line.strip().split("\t")

            if len(fields) < 12:
                continue

            read_id = fields[0]
            target_name = ref_name

            matching_bases = int(fields[9])
            alignment_length = int(fields[10])
            mapq = int(fields[11])

            if alignment_length == 0:
                continue

            identity = matching_bases / alignment_length

            # Score based on alignment identity and mapping quality
            score = identity * mapq
            #we are keeping only the best-scoring reference for each read
            if read_id not in minimap_best_results:
                minimap_best_results[read_id] = {
                    "predicted": target_name,
                    "identity": identity,
                    "mapq": mapq,
                    "score": score
                }

            elif score > minimap_best_results[read_id]["score"]:
                minimap_best_results[read_id] = {
                    "predicted": target_name,
                    "identity": identity,
                    "mapq": mapq,
                    "score": score
                }

# MINIMAP2 REPORT

TOTAL_READS = len(kmer_results)

minimap_counts = defaultdict(int)
minimap_correct = 0
minimap_total = 0

for read_id, result in minimap_best_results.items():

    actual = read_id.split("|")[0]#extract true species from read ID (in cell 2 from record.id )
    predicted = result["predicted"]

    minimap_total += 1
    minimap_counts[predicted] += 1

    if predicted == actual:
        minimap_correct += 1

#aligned-only accuracy, coverage accuracy, and global accuracy
minimap_accuracy_aligned = (
    minimap_correct / minimap_total * 100
    if minimap_total > 0 else 0
)

minimap_coverage = (
    minimap_total / TOTAL_READS * 100
    if TOTAL_READS > 0 else 0
)

minimap_accuracy_global = (
    minimap_correct / TOTAL_READS * 100
    if TOTAL_READS > 0 else 0
)

print("\nMINIMAP2 CLASSIFICATION REPORT")
print(f"Total reads in metagenome: {TOTAL_READS}")
print(f"Reads with valid alignments: {minimap_total}")
print(f"Correctly classified aligned reads: {minimap_correct}")

print(f"\nAccuracy on aligned reads: "
      f"{minimap_accuracy_aligned:.2f}%")

print(f"Coverage: "
      f"{minimap_coverage:.2f}%")

print(f"Global accuracy: "
      f"{minimap_accuracy_global:.2f}%")

print("\nMINIMAP2 METAGENOMIC SAMPLE COMPOSITION")

for genome, count in sorted(minimap_counts.items()):
    print(f"{genome}: {count} reads")

minimap_end_time = time.time()

minimap_runtime = minimap_end_time - minimap_start_time

print(f"\nExecution time: {minimap_runtime:.2f} seconds")

Running minimap2 for each reference genome...
Running minimap2 for escherichia_coli...
Running minimap2 for streptococcus_infantis...
Running minimap2 for salmonella_enterica...
Running minimap2 for vibrio_mimicus...
Running minimap2 for bacillus_lentus...

MINIMAP2 CLASSIFICATION REPORT
Total reads in metagenome: 25000
Reads with valid alignments: 22394
Correctly classified aligned reads: 22116

Accuracy on aligned reads: 98.76%
Coverage: 89.58%
Global accuracy: 88.46%

MINIMAP2 METAGENOMIC SAMPLE COMPOSITION
bacillus_lentus: 4717 reads
escherichia_coli: 4392 reads
salmonella_enterica: 4185 reads
streptococcus_infantis: 4730 reads
vibrio_mimicus: 4370 reads

Execution time: 30.94 seconds


In [7]:
# K-MER VS MINIMAP2 COMPARISON

from collections import defaultdict
import pandas as pd

kmer_dict = {}
actual_dict = {}

for result in kmer_results:
    kmer_dict[result["read_id"]] = result["predicted"]
    actual_dict[result["read_id"]] = result["actual"]


common_reads = 0
same_predictions = 0

kmer_correct_common = 0
minimap_correct_common = 0

kmer_confusion = defaultdict(lambda: defaultdict(int))
minimap_confusion = defaultdict(lambda: defaultdict(int))

for read_id, minimap_result in minimap_best_results.items():

    if read_id not in kmer_dict:
        continue

    actual = actual_dict[read_id]

    kmer_pred = kmer_dict[read_id]
    minimap_pred = minimap_result["predicted"]

    common_reads += 1

    if kmer_pred == minimap_pred:
        same_predictions += 1

    if kmer_pred == actual:
        kmer_correct_common += 1

    if minimap_pred == actual:
        minimap_correct_common += 1

    kmer_confusion[actual][kmer_pred] += 1
    minimap_confusion[actual][minimap_pred] += 1


agreement = (
    same_predictions / common_reads * 100
    if common_reads > 0 else 0
)

kmer_accuracy_common = (
    kmer_correct_common / common_reads * 100
    if common_reads > 0 else 0
)

minimap_accuracy_common = (
    minimap_correct_common / common_reads * 100
    if common_reads > 0 else 0
)

print("\nK-MER VS MINIMAP2 COMPARISON")
print(f"Common reads compared: {common_reads}")
print(f"Same predictions: {same_predictions}")
print(f"Agreement: {agreement:.2f}%")

print(f"\nK-mer accuracy on common reads: "
      f"{kmer_accuracy_common:.2f}%")

print(f"Minimap2 accuracy on common reads: "
      f"{minimap_accuracy_common:.2f}%")


#K-MER CONFUSION MATRIX
labels = sorted(
    set(kmer_confusion.keys()) |
    {p for row in kmer_confusion.values() for p in row}
)

matrix = []

for actual in labels:

    row = []

    for predicted in labels:
        row.append(kmer_confusion[actual][predicted])

    matrix.append(row)

kmer_confusion_df = pd.DataFrame(
    matrix,
    index=labels,
    columns=labels
)

print("\nK-MER CONFUSION MATRIX")
display(kmer_confusion_df)


#MINIMAP2 CONFUSION MATRIX

labels = sorted(
    set(minimap_confusion.keys()) |
    {p for row in minimap_confusion.values() for p in row}
)

matrix = []

for actual in labels:

    row = []

    for predicted in labels:
        row.append(minimap_confusion[actual][predicted])

    matrix.append(row)

minimap_confusion_df = pd.DataFrame(
    matrix,
    index=labels,
    columns=labels
)

print("\nMINIMAP2 CONFUSION MATRIX")
display(minimap_confusion_df)


K-MER VS MINIMAP2 COMPARISON
Common reads compared: 22394
Same predictions: 18438
Agreement: 82.33%

K-mer accuracy on common reads: 82.84%
Minimap2 accuracy on common reads: 98.76%

K-MER CONFUSION MATRIX


,bacillus_lentus,escherichia_coli,salmonella_enterica,streptococcus_infantis,vibrio_mimicus
bacillus_lentus,4575,13,5,70,55
escherichia_coli,174,2984,801,15,391
salmonella_enterica,355,416,3281,16,110
streptococcus_infantis,734,2,4,3835,151
vibrio_mimicus,233,59,19,219,3877



MINIMAP2 CONFUSION MATRIX


,bacillus_lentus,escherichia_coli,salmonella_enterica,streptococcus_infantis,vibrio_mimicus
bacillus_lentus,4711,0,1,6,0
escherichia_coli,0,4255,108,1,1
salmonella_enterica,2,108,4066,0,2
streptococcus_infantis,2,0,7,4717,0
vibrio_mimicus,2,29,3,6,4367
